# MobiML Preprocessing Demo

In [ ]:
import sys

sys.path.append("../src")

import pandas as pd
import geopandas as gpd
from datetime import datetime

from mobiml.utils import convert_wgs_to_utm
from mobiml.datasets import AISDK, PreprocessedAISDK, BrestAIS
from mobiml.preprocessing import StationaryClientExtractor, Normalizer

## Normalizer

Normalize latitude, longitude, speed and direction values

In [ ]:
ais = BrestAIS(r"./data/nari_dynamic.csv", nrows=1000)
ais.df.head()

In [ ]:
ais_norm = Normalizer(ais).normalize(speed_max=5.0, replace=False)
ais_norm.df.head()

In [ ]:
normalized = Normalizer(ais).normalize(replace=True)
normalized.df.head()

## StationaryClientExtractor

Extracting AIS records around specific locations

In [ ]:
antennas = ['Point (11.96524 57.70730)', 'Point (11.63979 57.71941)', 'Point (11.78460 57.57255)']
antenna_radius_meters = 25000
epsg_code = convert_wgs_to_utm(11.96524, 57.70730)


def create_client_gdf(clients, client_radius_meters) -> gpd.GeoDataFrame:
    ids = [{'client': i} for i in range(len(clients))]
    df = pd.DataFrame(ids)
    df['geometry'] = gpd.GeoSeries.from_wkt(clients)
    gdf = gpd.GeoDataFrame(df, geometry=df.geometry, crs=4326)
    gdf = gdf.to_crs(epsg_code)
    gdf['geometry'] = gdf.buffer(client_radius_meters)
    return gdf.to_crs(4326)


buffered_antennas = create_client_gdf(antennas, antenna_radius_meters)
buffered_antennas.explore()

In [ ]:
min_lon, min_lat, max_lon, max_lat = buffered_antennas.geometry.total_bounds
aisdk = AISDK('./data/aisdk_20180208_sample.zip', min_lon, min_lat, max_lon, max_lat)
aisdk.plot()

In [ ]:
print(f"{datetime.now()} Extracting client data ...")
client_data = StationaryClientExtractor(aisdk).extract(buffered_antennas)

In [ ]:
out_path = './data/ais-extracted-stationary.feather'
print(f"{datetime.now()} Writing output to {out_path}")
client_data.to_feather(out_path)

### Reading extracted AIS records from file

In [ ]:
aisdk = PreprocessedAISDK(out_path)
aisdk.df

In [ ]:
aisdk.plot()